### Generate human annotation excel and original backup

In [ ]:
import json
import random
import pandas as pd
from collections import defaultdict

# 1. Load 1505 JSON records
with open('./ris/refined/classification_results_deepseek_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Stratify and group by (domain, technology) combinations
stratified_groups = defaultdict(list)
for item in data:
    domain = item['classification']['domain']
    tech = item['classification']['technology']
    
    #  If domain is a list, convert it to a comma-separated string
    if isinstance(domain, list):
        domain = ", ".join(sorted([str(d) for d in domain]))
    else:
        domain = str(domain)
        
    #  If tech is a list, convert it to a comma-separated string as well
    if isinstance(tech, list):
        tech = ", ".join(sorted([str(t) for t in tech]))
    else:
        tech = str(tech)

    # Now both domain and tech are strings, so tuple keys are valid
    joint_key = (domain, tech)
    stratified_groups[joint_key].append(item)

# 3. Sample a total of 100 papers proportionally
sampled_data = []
total_papers = len(data)
target_total_sample = 100

for joint_key, papers in list(stratified_groups.items()):
    # Calculate the sample size for this combination based on its proportion
    sample_size = round((len(papers) / total_papers) * target_total_sample)
    
    # Ensure at least one sample is selected if the group exists
    if sample_size == 0 and len(papers) > 0:
        sample_size = 1
        
    if len(papers) < sample_size:
        sample_size = len(papers)
    
    # Randomly sample from the group
    sampled_papers = random.sample(papers, sample_size)
    sampled_data.extend(sampled_papers)

# 4. Adjust if rounding causes the total sample count to differ from 100
if len(sampled_data) > target_total_sample:
    sampled_data = random.sample(sampled_data, target_total_sample)
elif len(sampled_data) < target_total_sample:
    # Fill the remaining quota from unselected records
    sampled_ids = {item['record_index'] for item in sampled_data}
    remaining_pool = [item for item in data if item['record_index'] not in sampled_ids]
    needed = target_total_sample - len(sampled_data)
    sampled_data.extend(random.sample(remaining_pool, needed))

# 5. Shuffle the order to avoid labeling bias
random.shuffle(sampled_data)

# 6. Export the blind review Excel file (all-English columns, without DeepSeek labels)
blind_review_list = []
for item in sampled_data:
    blind_review_list.append({
        "record_index": item["record_index"],
        "title": item["title"],
        "abstract": item["abstract"],
        "human_domain": "",       
        "human_technology": ""    
    })

df_blind = pd.DataFrame(blind_review_list)
df_blind.to_excel("./ris/refined/blind_human_annotation.xlsx", index=False)


ds_answers = []
for item in sampled_data:
    ds_answers.append({
        "record_index": item["record_index"],
        "ds_domain": item["classification"]["domain"],
        "ds_technology": item["classification"]["technology"]
    })
df_ds = pd.DataFrame(ds_answers)
df_ds.to_excel("./ris/refined/deepseek_answers_backup.xlsx", index=False)

print(f"Successfully sampled {len(df_blind)} papers.")
print("'blind_human_annotation.xlsx' and 'deepseek_answers_backup.xlsx' have been generated.")

Successfully sampled 100 papers.
'blind_human_annotation.xlsx' and 'deepseek_answers_backup.xlsx' have been generated.


### Calculate Cohen's Kappa

In [ ]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd
import ast
import re

# 1. Read the completed blind review file and the backup DeepSeek labels
df_human = pd.read_excel("./ris/refined/blind_human_annotation.xlsx")
df_ds = pd.read_excel("./ris/refined/deepseek_answers_backup.xlsx")

# 2. Align the two records using record_index
df_merged = pd.merge(df_human, df_ds, on="record_index")

# 2.1 Exclude rows where comment is not empty; these rows are not included in metric calculation
if "comment" in df_merged.columns:
    df_merged["comment"] = df_merged["comment"].fillna("").astype(str).str.strip()
    df_merged = df_merged[df_merged["comment"] == ""].copy()

# 2.2 Normalize possible list labels, convert to lowercase, and support partial matches between lists
def normalize_label_value(value):
    if isinstance(value, list):
        items = [str(v).strip().lower() for v in value if str(v).strip()]
        return [item for item in items if item]
    if pd.isna(value):
        return []
    text = str(value).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                items = [str(v).strip().lower() for v in parsed if str(v).strip()]
                return [item for item in items if item]
        except Exception:
            pass
    parts = re.split(r"[;,/|]\s*", text)
    if len(parts) > 1:
        items = [part.strip().lower() for part in parts if part.strip()]
        return [item for item in items if item]
    return [text.lower()]


def choose_label(human_value, ds_value):
    human_items = normalize_label_value(human_value)
    ds_items = normalize_label_value(ds_value)
    if human_items and ds_items:
        overlap = sorted(set(human_items) & set(ds_items))
        if overlap:
            return overlap[0], overlap[0]
    human_label = human_items[0] if human_items else "missing"
    ds_label = ds_items[0] if ds_items else "missing"
    return human_label, ds_label

normalized_domains = df_merged.apply(
    lambda row: choose_label(row["human_domain"], row["ds_domain"]), axis=1
)
df_merged[["human_domain", "ds_domain"]] = pd.DataFrame(normalized_domains.tolist(), index=df_merged.index)

normalized_techs = df_merged.apply(
    lambda row: choose_label(row["human_technology"], row["ds_technology"]), axis=1
)
df_merged[["human_technology", "ds_technology"]] = pd.DataFrame(normalized_techs.tolist(), index=df_merged.index)

# 3. Extract annotation vectors for both labels
y_human_domain = df_merged["human_domain"]
y_ds_domain = df_merged["ds_domain"]

y_human_tech = df_merged["human_technology"]
y_ds_tech = df_merged["ds_technology"]

# 4. Compute Cohen's Kappa
kappa_domain = cohen_kappa_score(y_human_domain, y_ds_domain)
kappa_tech = cohen_kappa_score(y_human_tech, y_ds_tech)

# 5. Calculate the proportion requiring human re-annotation or correction
mismatch_domain = (y_human_domain != y_ds_domain).sum() / len(df_merged)
mismatch_tech = (y_human_tech != y_ds_tech).sum() / len(df_merged)

# 6. Print the results in English to the console
print("=== Evaluation Results ===")
print(f"Domain Dimension - Cohen's Kappa: {kappa_domain:.4f}")
print(f"Domain Dimension - Re-annotation Proportion: {mismatch_domain * 100:.2f}%")
print("-" * 30)
print(f"Technology Dimension - Cohen's Kappa: {kappa_tech:.4f}")
print(f"Technology Dimension - Re-annotation Proportion: {mismatch_tech * 100:.2f}%")


=== Evaluation Results ===
Domain Dimension - Cohen's Kappa: 0.6443
Domain Dimension - Re-annotation Proportion: 34.09%
------------------------------
Technology Dimension - Cohen's Kappa: 0.6309
Technology Dimension - Re-annotation Proportion: 30.68%


/Users/a/opt/anaconda3/envs/hllmc/lib/python3.10/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/Users/a/opt/anaconda3/envs/hllmc/lib/python3.10/site-packages/sklearn/metrics/_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
/Users/a/opt/anaconda3/envs/hllmc/lib/python3.10/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
/Users/a/opt/anaconda3/envs/hllmc/lib/python3.10/site-pac

In [38]:
len(df_merged)

88

In [42]:
(y_human_domain != y_ds_domain).sum()

np.int64(30)